<a href="https://colab.research.google.com/github/iKarmen/proctor/blob/1-download-and-explore-small-language-model-evaluate-with-trainer-and-document-results/Train_your_First_Language_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Train a Language Models

## Before training lets explore a model with fewer parameters
Exploring a model before training can provide valuable insights

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M")

### Inspect Model Configuration
Examine the model's configuration (model.config) to understand its architecture, hyperparameters, and other settings like hidden layers, attention heads, and vocabulary size. This gives a high-level overview of the model's structure.

In [4]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "dtype": "bfloat16",
  "eos_token_id": 0,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 576,
  "initializer_range": 0.041666666666666664,
  "intermediate_size": 1536,
  "is_llama_config": true,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 9,
  "num_hidden_layers": 30,
  "num_key_value_heads": 3,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_interleaved": false,
  "rope_parameters": {
    "rope_theta": 100000,
    "rope_type": "default"
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "use_cache": true,
  "vocab_size": 49152
}

### Examine Model Architecture
Print the model object directly to see a summary of its layers and their order. This can help you understand the model's internal structure and components.

In [5]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

### Explore Tokenizer Details
Investigate the tokenizer object. Check its vocabulary size (tokenizer.vocab_size), special tokens (tokenizer.special_tokens_map), and how it encodes/decodes sample text. This is crucial for understanding how the model processes input and output.

In [8]:
tokenizer

TokenizersBackend(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstrip=False, lstrip=False, sing

In [6]:
tokenizer.vocab_size

49152

In [7]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

### Experiment with Text Generation Parameters
Modify the generation parameters beyond max_new_tokens. Experiment with do_sample, temperature, top_k, top_p, and num_beams to see how they influence the creativity, coherence, and diversity of the generated text. This helps in understanding the model's generative capabilities.

In [9]:
prompt = "The quick brown fox"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

#### Generation with Default Parameters
Generate text using the default max_new_tokens to establish a baseline. This will serve as a comparison point for other parameter changes.

In [15]:
outputs = model.generate(
    inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id
)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog.

The quick brown fox jumps over the lazy dog


#### Experiment with Temperature and Sampling
Generate text using do_sample=True and vary the temperature (e.g., 0.5 for less randomness, 1.0 for more randomness) to observe its effect on creativity and coherence.

In [16]:
outputs = model.generate(
    inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.5
)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

The quick brown fox jumped over the lazy dog.

The dog jumped over the lazy cat.

The cat jumped over the dog.

It's a pretty good story, but it's not a good story.

The dog is pretty, and the cat is pretty.

The dog is not very, and the cat is very, and the dog is not very.

It's not really a story, but it's a good one.

The dog is not very,


#### Experiment with Top-K Sampling
Generate text using do_sample=True and top_k (e.g., 50, 10) to restrict token sampling to the top-k most probable words, controlling diversity.

In [17]:
outputs = model.generate(
    inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.5,
    top_k=50
)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

The quick brown fox jumped over the fence.

My father said he was going to take me to the doctor.

In the evening, my father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father went to the doctor.

My father


#### Experiment with Top-P (Nucleus) Sampling
Generate text using do_sample=True and top_p (e.g., 0.9, 0.5) to select tokens from the smallest set whose cumulative probability exceeds p, offering a flexible control over diversity.

In [24]:
outputs = model.generate(
    inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.1, #e.g., 0.5 for less randomness, 1.0 for more randomness
    top_k=10,
    top_p=0.9
)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps over the lazy dog

The quick brown fox jumps


## Train the Model

### Training Example
Fine-tune the "HuggingFaceTB/SmolLM2-135M" model using the Hugging Face `Trainer` API. Prepare a dummy dataset consisting of the sentences "The capital of France is Paris." and "Python is a programming language.", tokenize these sentences with a maximum length of 128, and run a training loop for 1 epoch with a batch size of 1 using the `TrainingArguments` and `DataCollatorForLanguageModeling`.

Prepare the dataset, configure training arguments, and initialize the Trainer for the SmolLM2-135M model.


In [26]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Fix: Assign pad_token to tokenizer to avoid ValueError
tokenizer.pad_token = tokenizer.eos_token

# 2. Create dummy data
data = {"text": ["The capital of France is Paris.", "Python is a programming language."]}

# 3. Convert to Hugging Face Dataset
dataset = Dataset.from_dict(data)

# 4. Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# 5. Map tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 6. Instantiate TrainingArguments
training_args = TrainingArguments(
    output_dir="./smollm-trainer",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

# 7. Initialize Trainer
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# 8. Start training
trainer.train()

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.679255
2,1.652170


TrainOutput(global_step=2, training_loss=2.1657127141952515, metrics={'train_runtime': 117.8264, 'train_samples_per_second': 0.017, 'train_steps_per_second': 0.017, 'total_flos': 163128508416.0, 'train_loss': 2.1657127141952515, 'epoch': 1.0})

## Evaluate the trained Model
Evaluate the fine-tuned "HuggingFaceTB/SmolLM2-135M" model by preparing an evaluation dataset with the sentences "The capital of France is Paris." and "Python is a programming language.", then use `trainer.evaluate()` to calculate the loss. Additionally, perform qualitative inference by generating text from the prompt "The capital of France is" to verify if the model has successfully learned the training data, and summarize the results.

### Prepare Evaluation Dataset
Create and tokenize a dataset for evaluating the fine-tuned model using the same sentences as the training phase.


In [27]:
# 1. Create a dictionary named eval_data
eval_data = {"text": ["The capital of France is Paris.", "Python is a programming language."]}

# 2. Convert to Hugging Face Dataset object
from datasets import Dataset
eval_hf_dataset = Dataset.from_dict(eval_data)

# 3. & 4. Apply existing tokenize_function and store in eval_dataset
eval_dataset = eval_hf_dataset.map(tokenize_function, batched=True)

print(f"Evaluation dataset created with {len(eval_dataset)} samples.")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Evaluation dataset created with 2 samples.
